# ДЗ 4.

Датасет: Brazilian E-Commerce Public Dataset by Olist
https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce

Беру 6 таблиць: customers, orders, order_items, products, sellers, order_reviews.

## Завдання 1. Setup і typed-схема Olist

pgserver через звичайний pip у Colab не ставиться (є тільки для Python до 3.12), тому ставлю його окремо в папку `pg_pkg`.

In [ ]:
%pip install -q psycopg2-binary sqlalchemy pandas kagglehub fasteners platformdirs psutil
%pip install -q --upgrade --no-deps --only-binary=:all: --python-version 3.12 --target pg_pkg pgserver

In [ ]:
import os
import sys
import tempfile
from urllib.request import urlretrieve

sys.path.insert(0, os.path.abspath('pg_pkg'))

import kagglehub
import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

/usr/local/lib/python3.13/dist-packages/platformdirs/api.py:459: RuntimeDirWarning: XDG_RUNTIME_DIR is not set, falling back to /tmp/runtime-0
  return Path(self.user_runtime_dir)


Драйвер `psycopg2` пишу в адресі явно, інакше SQLAlchemy шукає psycopg 3.

In [ ]:
pg_dir = os.path.join(tempfile.gettempdir(), 'hw4_pg')
pg = pgserver.get_server(pg_dir, cleanup_mode='stop')
db_uri = pg.get_uri().replace('postgresql://', 'postgresql+psycopg2://')
engine = create_engine(db_uri, future=True)

with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar_one())

PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


Функція для запитів. Через `engine.begin()` зміни комітяться, а `pd.read_sql` для INSERT/UPDATE з RETURNING не підходить, бо робить rollback.

In [ ]:
def run(query):
    with engine.begin() as conn:
        res = conn.execute(text(query))
        if res.returns_rows:
            return pd.DataFrame(res.fetchall(), columns=list(res.keys()))
        print('ok, rows affected:', res.rowcount)

### Завантаження CSV

Спочатку KaggleHub. Якщо не вийде, качаю ті самі файли з офіційного GitHub Olist у `data/olist`.

In [ ]:
files = {
    'olist_customers_raw': 'olist_customers_dataset.csv',
    'olist_orders_raw': 'olist_orders_dataset.csv',
    'olist_order_items_raw': 'olist_order_items_dataset.csv',
    'olist_products_raw': 'olist_products_dataset.csv',
    'olist_sellers_raw': 'olist_sellers_dataset.csv',
    'olist_order_reviews_raw': 'olist_order_reviews_dataset.csv',
}

try:
    dataset_dir = kagglehub.dataset_download('olistbr/brazilian-ecommerce')
except Exception as e:
    print('KaggleHub не спрацював:', e)
    dataset_dir = 'data/olist'
    os.makedirs(dataset_dir, exist_ok=True)
    base_url = 'https://raw.githubusercontent.com/olist/work-at-olist-data/master/datasets/'
    for file_name in files.values():
        path = os.path.join(dataset_dir, file_name)
        if not os.path.exists(path):
            urlretrieve(base_url + file_name, path)

print(dataset_dir)

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
/kaggle/input/brazilian-ecommerce


Raw-таблиці (проміжний шар).

In [ ]:
raw_counts = {}

for table_name, file_name in files.items():
    path = os.path.join(dataset_dir, file_name)
    df = pd.read_csv(path)
    df.to_sql(table_name, engine, if_exists='replace', index=False, chunksize=10_000)
    raw_counts[table_name] = len(df)
    print(table_name, df.shape)

olist_customers_raw (99441, 5)
olist_orders_raw (99441, 8)
olist_order_items_raw (112650, 7)
olist_products_raw (32951, 9)
olist_sellers_raw (3095, 4)
olist_order_reviews_raw (99224, 7)


### Typed-таблиці

In [ ]:
run("""
DROP TABLE IF EXISTS customer_segments    CASCADE;
DROP TABLE IF EXISTS seller_score         CASCADE;
DROP TABLE IF EXISTS seller_alerts        CASCADE;
DROP TABLE IF EXISTS olist_customer_dim   CASCADE;
DROP TABLE IF EXISTS olist_order_reviews  CASCADE;
DROP TABLE IF EXISTS olist_order_items    CASCADE;
DROP TABLE IF EXISTS olist_orders         CASCADE;
DROP TABLE IF EXISTS olist_customers      CASCADE;
DROP TABLE IF EXISTS olist_products       CASCADE;
DROP TABLE IF EXISTS olist_sellers        CASCADE;

CREATE TABLE olist_customers (
    customer_id              TEXT PRIMARY KEY,
    customer_unique_id       TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city            TEXT,
    customer_state           CHAR(2) NOT NULL
);

CREATE INDEX idx_olist_customers_unique_id
    ON olist_customers (customer_unique_id);

CREATE TABLE olist_orders (
    order_id                      TEXT PRIMARY KEY,
    customer_id                   TEXT NOT NULL REFERENCES olist_customers(customer_id),
    order_status                  TEXT NOT NULL,
    order_purchase_timestamp      TIMESTAMP NOT NULL,
    order_approved_at             TIMESTAMP,
    order_delivered_carrier_date  TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN (
        'created', 'approved', 'invoiced', 'processing',
        'shipped', 'delivered', 'unavailable', 'canceled'
    ))
);

CREATE TABLE olist_products (
    product_id                 TEXT PRIMARY KEY,
    product_category_name      TEXT,
    product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id              TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city            TEXT,
    seller_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id            TEXT NOT NULL REFERENCES olist_orders(order_id),
    order_item_id       INTEGER NOT NULL,
    product_id          TEXT REFERENCES olist_products(product_id),
    seller_id           TEXT REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
    freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);

CREATE INDEX idx_olist_order_items_seller_order
    ON olist_order_items (seller_id, order_id);

CREATE TABLE olist_order_reviews (
    review_row_id            BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    review_id                TEXT,
    order_id                 TEXT NOT NULL REFERENCES olist_orders(order_id),
    review_score             SMALLINT CHECK (review_score BETWEEN 1 AND 5),
    review_comment_title     TEXT,
    review_comment_message   TEXT,
    review_creation_date     TIMESTAMP,
    review_answer_timestamp  TIMESTAMP
);

CREATE INDEX idx_olist_order_reviews_order_id
    ON olist_order_reviews (order_id);
""")

ok, rows affected: -1


raw -> typed. Спочатку таблиці без FK, потім orders, потім items і reviews. У CSV колонки товарів названі з помилкою `lenght`.

In [ ]:
run("""
INSERT INTO olist_customers
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix::INTEGER,
    customer_city,
    customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_products
SELECT
    product_id,
    product_category_name,
    product_name_lenght::INTEGER,
    product_description_lenght::INTEGER,
    product_photos_qty::INTEGER,
    product_weight_g::INTEGER,
    product_length_cm::INTEGER,
    product_height_cm::INTEGER,
    product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT
    seller_id,
    seller_zip_code_prefix::INTEGER,
    seller_city,
    seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_orders
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp::TIMESTAMP,
    order_approved_at::TIMESTAMP,
    order_delivered_carrier_date::TIMESTAMP,
    order_delivered_customer_date::TIMESTAMP,
    order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_order_items
SELECT
    order_id,
    order_item_id::INTEGER,
    product_id,
    seller_id,
    shipping_limit_date::TIMESTAMP,
    price::NUMERIC(12, 2),
    freight_value::NUMERIC(12, 2)
FROM olist_order_items_raw;

INSERT INTO olist_order_reviews (
    review_id,
    order_id,
    review_score,
    review_comment_title,
    review_comment_message,
    review_creation_date,
    review_answer_timestamp
)
SELECT
    review_id,
    order_id,
    review_score::SMALLINT,
    review_comment_title,
    review_comment_message,
    review_creation_date::TIMESTAMP,
    review_answer_timestamp::TIMESTAMP
FROM olist_order_reviews_raw;
""")

ok, rows affected: 99224


### Smoke-test

In [ ]:
typed = run("""
SELECT 'customers' AS table_name, COUNT(*) AS n_rows FROM olist_customers
UNION ALL SELECT 'orders', COUNT(*) FROM olist_orders
UNION ALL SELECT 'order_items', COUNT(*) FROM olist_order_items
UNION ALL SELECT 'products', COUNT(*) FROM olist_products
UNION ALL SELECT 'sellers', COUNT(*) FROM olist_sellers
UNION ALL SELECT 'reviews', COUNT(*) FROM olist_order_reviews
ORDER BY table_name;
""")

typed['raw_rows'] = typed['table_name'].map({
    'customers': raw_counts['olist_customers_raw'],
    'orders': raw_counts['olist_orders_raw'],
    'order_items': raw_counts['olist_order_items_raw'],
    'products': raw_counts['olist_products_raw'],
    'sellers': raw_counts['olist_sellers_raw'],
    'reviews': raw_counts['olist_order_reviews_raw'],
})
typed['diff'] = typed['raw_rows'] - typed['n_rows']
typed

,table_name,n_rows,raw_rows,diff
0,customers,99441,99441,0
1,order_items,112650,112650,0
2,orders,99441,99441,0
3,products,32951,32951,0
4,reviews,99224,99224,0
5,sellers,3095,3095,0


**Висновок:** у typed стільки ж рядків, скільки в raw, нічого не відкинулось. Відгуків 99 224, бо частина коментарів займає кілька рядків у файлі.

Кілька перевірок якості даних:

In [ ]:
run("""
SELECT
    (SELECT COUNT(*) FROM olist_customers) AS n_customer_id,
    (SELECT COUNT(DISTINCT customer_unique_id) FROM olist_customers) AS n_customer_unique_id,
    (SELECT COUNT(*) FROM olist_products WHERE product_category_name IS NULL) AS products_no_category,
    (SELECT COUNT(*) - COUNT(DISTINCT review_id) FROM olist_order_reviews) AS duplicate_review_id,
    (SELECT COUNT(*) FROM olist_order_reviews WHERE review_comment_message IS NULL) AS reviews_no_text,
    (SELECT COUNT(*) FROM olist_orders AS o
      WHERE NOT EXISTS (SELECT 1 FROM olist_order_items AS oi WHERE oi.order_id = o.order_id)) AS orders_no_items;
""")

,n_customer_id,n_customer_unique_id,products_no_category,duplicate_review_id,reviews_no_text,orders_no_items
0,99441,96096,610,814,58247,775


In [ ]:
run("""
SELECT order_status, COUNT(*) AS n_orders
FROM olist_orders
GROUP BY order_status
ORDER BY n_orders DESC;
""")

,order_status,n_orders
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


**Висновок:**
- customer_id 99 441, а реальних покупців (customer_unique_id) 96 096
- 610 товарів без категорії
- 814 повторів review_id, тому PK зроблено через review_row_id
- 58 247 відгуків без тексту
- 775 замовлень без позицій
- 97% замовлень delivered, є дивні статуси created (5) і approved (2)

## Завдання 2. DML

### 2.1. INSERT ... RETURNING

In [ ]:
run("""
DROP TABLE IF EXISTS seller_alerts CASCADE;

CREATE TABLE seller_alerts (
    alert_id   BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    seller_id  TEXT NOT NULL REFERENCES olist_sellers(seller_id),
    alert_type TEXT NOT NULL CHECK (alert_type IN ('late_delivery', 'low_rating', 'data_quality')),
    severity   SMALLINT NOT NULL CHECK (severity BETWEEN 1 AND 5),
    details    TEXT,
    created_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
);
""")

ok, rows affected: -1


In [ ]:
run("""
WITH seller_orders AS (
    SELECT DISTINCT
        oi.seller_id,
        oi.order_id
    FROM olist_order_items AS oi
),
late_sellers AS (
    SELECT
        so.seller_id,
        COUNT(DISTINCT o.order_id) AS late_orders
    FROM seller_orders AS so
    JOIN olist_orders AS o
        ON o.order_id = so.order_id
    WHERE o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
      AND o.order_delivered_customer_date::DATE > o.order_estimated_delivery_date::DATE
    GROUP BY so.seller_id
    ORDER BY late_orders DESC, so.seller_id
    LIMIT 1
)
INSERT INTO seller_alerts (seller_id, alert_type, severity, details)
SELECT
    seller_id,
    'late_delivery',
    4,
    'Seller has ' || late_orders || ' late delivered orders in the loaded dataset'
FROM late_sellers
RETURNING alert_id, seller_id, alert_type, severity, details, created_at;
""")

,alert_id,seller_id,alert_type,severity,details,created_at
0,1,4a3ca9315b744ce9f8e9374361493884,late_delivery,4,Seller has 172 late delivered orders in the lo...,2026-10-28 10:57:53.757061+00:00


**Пояснення:** alert отримав продавець з 172 запізненими замовленнями. RETURNING одразу показує створений рядок з alert_id і created_at, тому окремий SELECT не потрібен, і це можна використати як audit-log. ON CONFLICT тут немає, тому таблицю перед вставкою перестворюю.

### 2.2. INSERT ... ON CONFLICT: seller_score

In [ ]:
run("""
DROP TABLE IF EXISTS seller_score CASCADE;

CREATE TABLE seller_score (
    seller_id   TEXT PRIMARY KEY REFERENCES olist_sellers(seller_id),
    avg_rating  NUMERIC(3, 2),
    n_reviews   INTEGER NOT NULL DEFAULT 0 CHECK (n_reviews >= 0),
    tier        TEXT CHECK (tier IN ('gold', 'silver', 'bronze')),
    updated_at  TIMESTAMPTZ NOT NULL DEFAULT NOW()
);
""")

ok, rows affected: -1


Запит у змінній, щоб запустити двічі.

In [ ]:
seller_score_sql = """
WITH seller_orders AS (
    SELECT DISTINCT
        oi.seller_id,
        oi.order_id
    FROM olist_order_items AS oi
),
review_per_order AS (
    SELECT
        rv.order_id,
        AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
),
seller_rating AS (
    SELECT
        so.seller_id,
        AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_rating,
        COUNT(rpo.order_id) AS n_reviews
    FROM seller_orders AS so
    LEFT JOIN review_per_order AS rpo
        ON rpo.order_id = so.order_id
    GROUP BY so.seller_id
)
INSERT INTO seller_score (seller_id, avg_rating, n_reviews, tier)
SELECT
    seller_id,
    avg_rating,
    n_reviews,
    CASE
        WHEN avg_rating IS NULL THEN NULL
        WHEN avg_rating >= 4.5 THEN 'gold'
        WHEN avg_rating >= 3.5 THEN 'silver'
        ELSE 'bronze'
    END AS tier
FROM seller_rating
ON CONFLICT (seller_id) DO UPDATE
SET avg_rating = EXCLUDED.avg_rating,
    n_reviews  = EXCLUDED.n_reviews,
    tier       = EXCLUDED.tier,
    updated_at = NOW()
RETURNING seller_id, avg_rating, n_reviews, tier, updated_at;
"""

result = run(seller_score_sql)
print(len(result))
result.head(10)

3095


,seller_id,avg_rating,n_reviews,tier,updated_at
0,a61cc04793308395a840807104365121,5.00,1,gold,2026-10-28 10:57:53.906979+00:00
1,6d2f2e3b539480db1e0842b3a4e32e6e,3.67,3,silver,2026-10-28 10:57:53.906979+00:00
2,c53bcd3be457a342a97e39e5a9f0be22,3.20,5,bronze,2026-10-28 10:57:53.906979+00:00
3,9591fc341b1bfb7ef561e2968ec6e011,1.00,1,bronze,2026-10-28 10:57:53.906979+00:00
4,f9ec7093df3a7b346b7bcf7864069ca3,4.78,9,gold,2026-10-28 10:57:53.906979+00:00
5,fc38b5dceee1a730fad8853453437fbd,4.60,5,gold,2026-10-28 10:57:53.906979+00:00
6,5def4c3732941a971cba8fdee992ede1,4.60,5,gold,2026-10-28 10:57:53.906979+00:00
7,b39d7fe263ef469605dbb32608aee0af,3.88,43,silver,2026-10-28 10:57:53.906979+00:00
8,ea65d8b58316a6f2362f2a9e4b3e86ad,4.00,9,silver,2026-10-28 10:57:53.906979+00:00
9,a45765f8afb1e594b22bf9e974b46765,4.00,1,silver,2026-10-28 10:57:53.906979+00:00


Другий запуск:

In [ ]:
result2 = run(seller_score_sql)
print('returned:', len(result2))

run("""
SELECT
    COUNT(*) AS n_rows,
    COUNT(DISTINCT seller_id) AS n_sellers,
    COUNT(*) FILTER (WHERE tier IS NULL) AS no_tier,
    COUNT(*) FILTER (WHERE tier = 'gold') AS gold,
    COUNT(*) FILTER (WHERE tier = 'silver') AS silver,
    COUNT(*) FILTER (WHERE tier = 'bronze') AS bronze
FROM seller_score;
""")

returned: 3095


,n_rows,n_sellers,no_tier,gold,silver,bronze
0,3095,3095,5,983,1568,539


Продавці без відгуків:

In [ ]:
run("""
SELECT seller_id, avg_rating, n_reviews, tier
FROM seller_score
WHERE n_reviews = 0
ORDER BY seller_id;
""")

,seller_id,avg_rating,n_reviews,tier
0,20f0aeea30bc3b8c4420be8ced4226c0,None,0,None
1,3820c6537b3853be1a0687d5a60807ec,None,0,None
2,400f221ab83037c0ea78f1267da9624d,None,0,None
3,80ceebb4ee9b31afb6c6a916a574a1e2,None,0,None
4,bcd2d7510d58e293f20fad6438c1b314,None,0,None


Порівняння з неправильним join через order_items:

In [ ]:
run("""
SELECT
    SUM(n_wrong) AS reviews_wrong_join,
    (SELECT SUM(n_reviews) FROM seller_score) AS reviews_correct
FROM (
    SELECT oi.seller_id, COUNT(rv.review_score) AS n_wrong
    FROM olist_order_items AS oi
    LEFT JOIN olist_order_reviews AS rv
        ON rv.order_id = oi.order_id
    GROUP BY oi.seller_id
) AS t;
""")

,reviews_wrong_join,reviews_correct
0,112372,99247


**Пояснення:**
- `SELECT DISTINCT` залишає одну пару seller × order, навіть якщо в замовленні кілька товарів продавця.
- Відгуки спочатку зводжу до одного значення на order_id (547 замовлень мають кілька відгуків). Неправильний join дає 112 372 відгуки, правильний 99 247.
- `ON CONFLICT (seller_id) DO UPDATE` оновлює існуючий рядок замість вставки дубля. Після другого запуску так само 3 095 рядків.
- У 5 продавців немає відгуків: AVG дає NULL, COUNT дає 0, tier теж NULL.

Обмеження: відгук прив'язаний до замовлення, а не до продавця. 1 278 замовлень мають кількох продавців, і оцінка йде кожному з них, тому 99 247 > 99 224.

### 2.3. UPDATE ... FROM: прапорець запізнення в olist_orders

In [ ]:
run("""
ALTER TABLE olist_orders
    ADD COLUMN IF NOT EXISTS is_late BOOLEAN NOT NULL DEFAULT FALSE;
""")

ok, rows affected: -1


RETURNING повертає всі замовлення, тому показую тільки кількість і перші рядки.

In [ ]:
updated = run("""
WITH delivery_flags AS (
    SELECT
        order_id,
        (
            order_delivered_customer_date IS NOT NULL
            AND order_estimated_delivery_date IS NOT NULL
            AND order_delivered_customer_date::DATE > order_estimated_delivery_date::DATE
        ) AS is_late_calc
    FROM olist_orders
)
UPDATE olist_orders AS o
SET is_late = f.is_late_calc
FROM delivery_flags AS f
WHERE f.order_id = o.order_id
RETURNING o.order_id, o.customer_id, o.is_late;
""")

print('updated rows:', len(updated))
print(updated['is_late'].value_counts())
updated.head()

updated rows: 99441
is_late
False    92906
True      6535
Name: count, dtype: int64


,order_id,customer_id,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,False


In [ ]:
run("""
SELECT
    is_late,
    COUNT(*) AS n_orders,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
FROM olist_orders
WHERE order_status = 'delivered'
GROUP BY is_late
ORDER BY is_late;
""")

,is_late,n_orders,pct
0,False,89944,93.23
1,True,6534,6.77


**Пояснення:** оновлено 99 441 рядок, запізнених 6 535 (6.77% серед delivered). Прапорець кожен раз рахується заново з дат, тому повторний запуск дає той самий результат.

### 2.4. DELETE ... RETURNING

In [ ]:
run("""
DELETE FROM seller_alerts
WHERE created_at < NOW() - INTERVAL '30 days'
RETURNING alert_id, seller_id, alert_type, created_at;
""")

,alert_id,seller_id,alert_type,created_at


In [ ]:
run("""
SELECT alert_id, seller_id, alert_type, created_at
FROM seller_alerts;
""")

,alert_id,seller_id,alert_type,created_at
0,1,4a3ca9315b744ce9f8e9374361493884,late_delivery,2026-10-28 10:57:53.757061+00:00


**Пояснення:** результат порожній, бо alert створений щойно і не старший за 30 днів. RETURNING показав би, що саме видалено.

## Завдання 3. customer_segments

Спочатку довідник покупців, один рядок на customer_unique_id.

In [ ]:
run("""
DROP TABLE IF EXISTS customer_segments CASCADE;
DROP TABLE IF EXISTS olist_customer_dim CASCADE;

CREATE TABLE olist_customer_dim (
    customer_unique_id TEXT PRIMARY KEY,
    latest_customer_id TEXT REFERENCES olist_customers(customer_id),
    customer_state     CHAR(2),
    customer_city      TEXT
);

INSERT INTO olist_customer_dim (
    customer_unique_id,
    latest_customer_id,
    customer_state,
    customer_city
)
SELECT DISTINCT ON (c.customer_unique_id)
    c.customer_unique_id,
    c.customer_id AS latest_customer_id,
    c.customer_state,
    c.customer_city
FROM olist_customers AS c
LEFT JOIN olist_orders AS o
    ON o.customer_id = c.customer_id
ORDER BY
    c.customer_unique_id,
    o.order_purchase_timestamp DESC NULLS LAST,
    c.customer_id;
""")

ok, rows affected: 96096


In [ ]:
run("""
CREATE TABLE customer_segments (
    segment_id          BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    customer_unique_id  TEXT NOT NULL,
    segment_name        TEXT NOT NULL
        CHECK (segment_name IN ('VIP', 'regular', 'new', 'inactive')),

    rfm_score           SMALLINT NOT NULL
        CHECK (rfm_score BETWEEN 1 AND 5),

    recency_days        INTEGER
        CHECK (recency_days IS NULL OR recency_days >= 0),

    monetary_value      NUMERIC(12, 2) NOT NULL DEFAULT 0
        CHECK (monetary_value >= 0),

    n_orders            INTEGER NOT NULL DEFAULT 0
        CHECK (n_orders >= 0),

    assigned_at         TIMESTAMPTZ NOT NULL DEFAULT NOW(),

    UNIQUE (customer_unique_id)
);

ALTER TABLE customer_segments
    ADD CONSTRAINT fk_customer_segments_customer_unique
    FOREIGN KEY (customer_unique_id)
    REFERENCES olist_customer_dim(customer_unique_id)
    ON DELETE CASCADE;
""")

ok, rows affected: -1


Наповнення через ON CONFLICT, теж двічі.

In [ ]:
segments_sql = """
WITH reference_date AS (
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
order_totals AS (
    SELECT
        o.order_id,
        o.customer_id,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(oi.price + oi.freight_value), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.order_id,
        o.customer_id,
        o.order_purchase_timestamp
),
customer_stats AS (
    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT ot.order_id) AS n_orders,
        COALESCE(SUM(ot.order_total), 0)::NUMERIC(12, 2) AS monetary_value,
        MAX(ot.order_date) AS last_order_date
    FROM olist_customer_dim AS cd
    JOIN olist_customers AS c
        ON c.customer_unique_id = cd.customer_unique_id
    LEFT JOIN order_totals AS ot
        ON ot.customer_id = c.customer_id
    GROUP BY c.customer_unique_id
),
segmented AS (
    SELECT
        cs.customer_unique_id,
        cs.n_orders,
        cs.monetary_value,
        CASE
            WHEN cs.last_order_date IS NULL THEN NULL
            ELSE (rd.as_of_date - cs.last_order_date)::INTEGER
        END AS recency_days,
        CASE
            WHEN cs.monetary_value >= 1000 THEN 'VIP'
            WHEN cs.last_order_date IS NOT NULL
                 AND (rd.as_of_date - cs.last_order_date)::INTEGER <= 90
                 AND cs.n_orders = 1 THEN 'new'
            WHEN cs.n_orders >= 2 OR cs.monetary_value >= 100 THEN 'regular'
            ELSE 'inactive'
        END AS segment_name,
        CASE
            WHEN cs.monetary_value >= 1000 THEN 5
            WHEN cs.monetary_value >= 500  THEN 4
            WHEN cs.n_orders >= 2          THEN 3
            WHEN cs.n_orders = 1           THEN 2
            ELSE 1
        END::SMALLINT AS rfm_score
    FROM customer_stats AS cs
    CROSS JOIN reference_date AS rd
)
INSERT INTO customer_segments (
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders
)
SELECT
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders
FROM segmented
ON CONFLICT (customer_unique_id) DO UPDATE
SET segment_name   = EXCLUDED.segment_name,
    rfm_score      = EXCLUDED.rfm_score,
    recency_days   = EXCLUDED.recency_days,
    monetary_value = EXCLUDED.monetary_value,
    n_orders       = EXCLUDED.n_orders,
    assigned_at    = NOW()
RETURNING
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders;
"""

seg = run(segments_sql)
print(len(seg))
seg.head(10)

96096


,customer_unique_id,segment_name,rfm_score,recency_days,monetary_value,n_orders
0,0000366f3b9a7992bf8c76cfdf3221e2,regular,2,161,141.90,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,inactive,2,164,27.19,1
2,0000f46a3911fa3c0805444483337064,inactive,2,587,86.22,1
3,0000f6ccb0745a6a4b88665a16c9f078,inactive,2,371,43.62,1
4,0004aac84e0df4da2b147fca70cf8255,regular,2,338,196.89,1
5,0004bd2a26a76fe21f786e4fbd80607f,regular,2,196,166.98,1
6,00050ab1314c0e55a6ca13cf7181fecf,inactive,2,181,35.38,1
7,00053a61a98854899e70ed204dd4bafe,regular,2,232,419.18,1
8,0005e1862207bf6ccc02e4228effd9a0,regular,2,593,150.12,1
9,0005ef4cd20d2893f0d9fbd94d3c0d97,regular,2,220,129.76,1


In [ ]:
seg2 = run(segments_sql)
print('returned on second run:', len(seg2))

run("""
SELECT
    (SELECT COUNT(*) FROM olist_customer_dim) AS n_customer_dim,
    COUNT(*) AS n_segments,
    MAX(segment_id) AS max_segment_id
FROM customer_segments;
""")

returned on second run: 96096


,n_customer_dim,n_segments,max_segment_id
0,96096,96096,96096


Розподіл по сегментах:

In [ ]:
run("""
SELECT
    segment_name,
    COUNT(*) AS n_customers,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct,
    ROUND(AVG(monetary_value), 2) AS avg_monetary,
    ROUND(AVG(n_orders), 2) AS avg_orders,
    ROUND(AVG(recency_days), 0) AS avg_recency_days
FROM customer_segments
GROUP BY segment_name
ORDER BY n_customers DESC;
""")

,segment_name,n_customers,pct,avg_monetary,avg_orders,avg_recency_days
0,regular,45982,47.85,224.57,1.07,307
1,inactive,40032,41.66,59.28,1.00,316
2,new,8891,9.25,140.39,1.00,72
3,VIP,1191,1.24,1591.81,1.11,290


Constraints:

In [ ]:
run("""
SELECT
    conname,
    contype,
    pg_get_constraintdef(oid) AS definition
FROM pg_constraint
WHERE conrelid = 'customer_segments'::regclass
ORDER BY conname;
""")

,conname,contype,definition
0,customer_segments_customer_unique_id_key,u,UNIQUE (customer_unique_id)
1,customer_segments_monetary_value_check,c,CHECK ((monetary_value >= (0)::numeric))
2,customer_segments_n_orders_check,c,CHECK ((n_orders >= 0))
3,customer_segments_pkey,p,PRIMARY KEY (segment_id)
4,customer_segments_recency_days_check,c,CHECK (((recency_days IS NULL) OR (recency_day...
5,customer_segments_rfm_score_check,c,CHECK (((rfm_score >= 1) AND (rfm_score <= 5)))
6,customer_segments_segment_name_check,c,"CHECK ((segment_name = ANY (ARRAY['VIP'::text,..."
7,fk_customer_segments_customer_unique,f,FOREIGN KEY (customer_unique_id) REFERENCES ol...


**Пояснення:**
- customer_id в Olist новий на кожне замовлення, тому рахую по customer_unique_id, інакше у всіх буде 1 замовлення.
- У olist_customers customer_unique_id не унікальний, тому для FK потрібна olist_customer_dim, де він PK.
- reference_date беру з даних (останнє замовлення + 1 день). З CURRENT_DATE recency був би 2000+ днів і результат мінявся б щодня.
- Завдяки UNIQUE і ON CONFLICT другий запуск оновлює рядки, а не додає: так само 96 096. Але лічильник identity все одно зсунувся до 192 192, тому id будуть з пропусками.

Regular 47.86%, inactive 41.65%, new 9.25%, VIP 1.24%. 1 099 з 1 192 VIP мають лише одне замовлення.

## Завдання 4. JOIN

### 4.1. INNER JOIN

**Business-question:** які категорії і штати продавців дають найбільший revenue у delivered-замовленнях?

In [ ]:
run("""
SELECT
    p.product_category_name,
    s.seller_state,
    COUNT(DISTINCT oi.order_id) AS n_orders,
    SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS revenue
FROM olist_order_items AS oi
JOIN olist_orders AS o
    ON o.order_id = oi.order_id
JOIN olist_products AS p
    ON p.product_id = oi.product_id
JOIN olist_sellers AS s
    ON s.seller_id = oi.seller_id
WHERE o.order_status = 'delivered'
GROUP BY p.product_category_name, s.seller_state
ORDER BY revenue DESC NULLS LAST
LIMIT 10;
""")

,product_category_name,seller_state,n_orders,revenue
0,cama_mesa_banho,SP,8214,1070048.14
1,relogios_presentes,SP,4494,1016168.80
2,beleza_saude,SP,5684,790967.66
3,esporte_lazer,SP,4772,680165.54
4,moveis_decoracao,SP,4597,605650.63
5,cool_stuff,SP,2621,484196.53
6,utilidades_domesticas,SP,4021,471810.66
7,automotivo,SP,3018,438833.22
8,informatica_acessorios,SP,2783,388506.42
9,bebes,SP,1981,339129.74


**Interpretation:** весь топ-10 це продавці з SP. Лідери cama_mesa_banho (1.07 млн) і relogios_presentes (1.02 млн). Виручка сильно залежить від одного штату.

### 4.2. LEFT JOIN + COALESCE

**Business-question:** які ознаки можна зібрати для кожного покупця?

In [ ]:
run("""
WITH order_totals AS (
    SELECT
        o.order_id,
        c.customer_unique_id,
        SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.order_id,
        c.customer_unique_id
),
review_per_order AS (
    SELECT
        rv.order_id,
        AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
),
customer_stats AS (
    SELECT
        ot.customer_unique_id,
        COUNT(DISTINCT ot.order_id) AS n_orders,
        COALESCE(SUM(ot.order_total), 0)::NUMERIC(12, 2) AS total_spend,
        AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_review_score
    FROM order_totals AS ot
    LEFT JOIN review_per_order AS rpo
        ON rpo.order_id = ot.order_id
    GROUP BY ot.customer_unique_id
)
SELECT
    cd.customer_unique_id,
    cd.customer_state,
    COALESCE(cs.n_orders, 0) AS n_orders,
    COALESCE(cs.total_spend, 0)::NUMERIC(12, 2) AS total_spend,
    COALESCE(cs.avg_review_score, 0)::NUMERIC(3, 2) AS avg_review_score
FROM olist_customer_dim AS cd
LEFT JOIN customer_stats AS cs
    ON cs.customer_unique_id = cd.customer_unique_id
ORDER BY total_spend DESC NULLS LAST
LIMIT 20;
""")

,customer_unique_id,customer_state,n_orders,total_spend,avg_review_score
0,0a0a92112bd4c708ca5fde585afaa872,RJ,1,13664.08,1.00
1,da122df9eeddfedc1dc1f5349a1a690c,RJ,2,7571.63,5.00
2,763c8b1c9c68a0229c42c9fc6f662b93,ES,1,7274.88,1.00
3,dc4802a71eae9be1dd28f5d788ceb526,MS,1,6929.31,5.00
4,459bef486812aa25204be022145caa62,ES,1,6922.21,0.00
5,ff4159b92c40ebe40454e3e6a7c35ed6,SP,1,6726.66,5.00
6,4007669dec559734d6f53e029e360987,MG,1,6081.54,1.00
7,5d0a2980b292d049061542014e8960bf,GO,1,4809.44,1.00
8,eebb5dda148d3893cdaf5b5ca3040ccb,SP,1,4764.34,4.00
9,48e1ac109decbb87765a3eade6854098,PB,1,4681.78,5.00


Покупці без відгуків:

In [ ]:
run("""
SELECT COUNT(*) AS customers_without_review
FROM olist_customer_dim AS cd
WHERE NOT EXISTS (
    SELECT 1
    FROM olist_customers AS c
    JOIN olist_orders AS o
        ON o.customer_id = c.customer_id
    JOIN olist_order_reviews AS rv
        ON rv.order_id = o.order_id
    WHERE c.customer_unique_id = cd.customer_unique_id
);
""")

,customers_without_review
0,716


**Interpretation:** у топі майже всі мають 1 замовлення, найбільше 13 664. review_per_order не дає помножити оцінку на кількість товарів. Але 716 покупців без відгуку отримують 0 через COALESCE, а для моделі це виглядає як погана оцінка. Краще NULL + ознака has_review.

### 4.3. FULL OUTER JOIN

**Business-question:** які штати є тільки у клієнтів, тільки у продавців або в обох?

In [ ]:
run("""
WITH customer_states AS (
    SELECT customer_state AS state, COUNT(*) AS n_customers
    FROM olist_customers
    GROUP BY customer_state
), seller_states AS (
    SELECT seller_state AS state, COUNT(*) AS n_sellers
    FROM olist_sellers
    GROUP BY seller_state
)
SELECT
    COALESCE(c.state, s.state) AS state,
    c.n_customers,
    s.n_sellers,
    CASE
        WHEN c.state IS NOT NULL AND s.state IS NOT NULL THEN 'both'
        WHEN c.state IS NOT NULL THEN 'customers_only'
        ELSE 'sellers_only'
    END AS state_presence
FROM customer_states AS c
FULL OUTER JOIN seller_states AS s
    ON s.state = c.state
ORDER BY state;
""")

,state,n_customers,n_sellers,state_presence
0,AC,81,1.0,both
1,AL,413,NaN,customers_only
2,AM,148,1.0,both
3,AP,68,NaN,customers_only
4,BA,3380,19.0,both
5,CE,1336,13.0,both
6,DF,2140,30.0,both
7,ES,2033,23.0,both
8,GO,2020,40.0,both
9,MA,747,1.0,both


**Interpretation:** клієнти є в 27 штатах, продавці в 23. AL, AP, RR, TO мають тільки клієнтів, `sellers_only` немає. У SP 1 849 продавців, а в багатьох штатах по 1-2.

### 4.4. SELF JOIN

**Business-question:** які пари продавців з одного штату, але різних міст можна порівнювати?

In [ ]:
run("""
SELECT
    s1.seller_id AS seller_a,
    s2.seller_id AS seller_b,
    s1.seller_state,
    s1.seller_city AS city_a,
    s2.seller_city AS city_b
FROM olist_sellers AS s1
JOIN olist_sellers AS s2
    ON s1.seller_state = s2.seller_state
   AND s1.seller_id < s2.seller_id
   AND s1.seller_city IS DISTINCT FROM s2.seller_city
ORDER BY s1.seller_state, seller_a, seller_b
LIMIT 20;
""")

,seller_a,seller_b,seller_state,city_a,city_b
0,1444c08e64d55fb3c25f0f09c07ffcf2,2b402d5dc42554061f8ea98d1916f148,BA,lauro de freitas,irece
1,1444c08e64d55fb3c25f0f09c07ffcf2,398cb257329ef7af7f1943a8974a3cbc,BA,lauro de freitas,salvador
2,1444c08e64d55fb3c25f0f09c07ffcf2,4221a7df464f1fe2955934e30ff3a5a1,BA,lauro de freitas,bahia
3,1444c08e64d55fb3c25f0f09c07ffcf2,43753b27d77860f1654aa72e251a7878,BA,lauro de freitas,barro alto
4,1444c08e64d55fb3c25f0f09c07ffcf2,4aba391bc3b88717ce08eb11e44937b2,BA,lauro de freitas,arraial d'ajuda (porto seguro)
5,1444c08e64d55fb3c25f0f09c07ffcf2,4fb41dff7c50136976d1a5cf004a42e2,BA,lauro de freitas,feira de santana
6,1444c08e64d55fb3c25f0f09c07ffcf2,651530bf5c607240ccdd89a30c9c9712,BA,lauro de freitas,ipira
7,1444c08e64d55fb3c25f0f09c07ffcf2,659e8466eb3ff1b0e8740d74fb7bbedd,BA,lauro de freitas,eunapolis
8,1444c08e64d55fb3c25f0f09c07ffcf2,75d34ebb1bd0bd7dde40dd507b8169c3,BA,lauro de freitas,salvador
9,1444c08e64d55fb3c25f0f09c07ffcf2,a3dd39f583bc80bd8c5901c95878921e,BA,lauro de freitas,salvador


Кількість пар по штатах:

In [ ]:
run("""
SELECT
    s1.seller_state,
    COUNT(*) AS n_pairs
FROM olist_sellers AS s1
JOIN olist_sellers AS s2
    ON s1.seller_state = s2.seller_state
   AND s1.seller_id < s2.seller_id
   AND s1.seller_city IS DISTINCT FROM s2.seller_city
GROUP BY s1.seller_state
ORDER BY n_pairs DESC
LIMIT 5;
""")

,seller_state,n_pairs
0,SP,1457751
1,PR,51665
2,MG,27184
3,SC,17208
4,RJ,10151


**Interpretation:** `s1.seller_id < s2.seller_id` прибирає дзеркальні пари. Тільки в SP 1.46 млн пар, тому порівнювати краще в межах конкретного штату. Ще видно помилку в даних: місто `bahia` (це назва штату).

### 4.5. Anti-join

**Business-question:** які замовлення не мають відгуку?

In [ ]:
run("""
SELECT
    o.order_id,
    o.customer_id,
    o.order_status,
    o.order_purchase_timestamp
FROM olist_orders AS o
LEFT JOIN olist_order_reviews AS rv
    ON rv.order_id = o.order_id
WHERE rv.review_row_id IS NULL
ORDER BY o.order_purchase_timestamp
LIMIT 20;
""")

,order_id,customer_id,order_status,order_purchase_timestamp
0,cda873529ca7ab71f677d5ec11a40304,76c74aaff2f3f7355f46d9818ad092b8,shipped,2016-10-05 16:57:30
1,5da2004d85f2828349d887c989da2adc,95f592086150f0c95d3082aea047b816,delivered,2016-10-05 20:20:59
2,02190241f7190a1f3c7e0df95a749c6a,55c9044401ffab0c717f24405e2e042f,delivered,2016-10-07 10:52:22
3,0efd0bc268d34da3f01f4ff25a6d4335,95446917717bb58d553d107d0f1668f6,shipped,2016-10-07 15:53:31
4,c776863a93dc0740c6e7d78104b21413,1710b798ebdc5db6665c815b90a64999,delivered,2016-10-10 15:01:27
5,d730d8d0ea3149120703933fdf98b2d4,ce2a66464da8b80dedd7639a6e489a58,shipped,2017-01-13 20:12:41
6,05cd156f0f47579a36d7a52724fc31b8,976224a95faf328b17654b1b1411d5e1,delivered,2017-01-14 18:27:28
7,719b940ab2f1e351ffb14a0a02ffe3c4,ede9a1daf58d256c0d648ef7e1479010,delivered,2017-01-16 13:01:51
8,fbd82b1fe98967de0db29b6cf134d00a,45f5c9d984dbfa2105e3544cc85133bf,delivered,2017-01-18 08:55:36
9,171398a0cd843c65ac78aee9e05a1f49,f0244902c07b7c0799e468218579a925,delivered,2017-01-18 16:07:37


In [ ]:
run("""
SELECT
    o.order_status,
    COUNT(*) AS n_orders_without_review
FROM olist_orders AS o
LEFT JOIN olist_order_reviews AS rv
    ON rv.order_id = o.order_id
WHERE rv.review_row_id IS NULL
GROUP BY o.order_status
ORDER BY n_orders_without_review DESC;
""")

,order_status,n_orders_without_review
0,delivered,646
1,shipped,75
2,canceled,20
3,unavailable,14
4,processing,6
5,invoiced,5
6,created,2


**Interpretation:** без відгуку 768 замовлень, з них 646 delivered. Для shipped це нормально, а для доставлених це пропуски, які треба враховувати в ознаках.

## Завдання 5. GROUP BY і HAVING

### 5.1. Revenue по категоріях (> 10 000)

In [ ]:
run("""
SELECT
    p.product_category_name,
    COUNT(DISTINCT oi.order_id) AS n_orders,
    SUM(oi.price)::NUMERIC(12, 2) AS gross_revenue,
    SUM(oi.freight_value)::NUMERIC(12, 2) AS freight,
    AVG(oi.price)::NUMERIC(8, 2) AS avg_item_price
FROM olist_order_items AS oi
JOIN olist_products AS p
    ON p.product_id = oi.product_id
GROUP BY p.product_category_name
HAVING SUM(oi.price) > 10000
ORDER BY gross_revenue DESC NULLS LAST;
""")

,product_category_name,n_orders,gross_revenue,freight,avg_item_price
0,beleza_saude,8836,1258681.34,182566.73,130.16
1,relogios_presentes,5624,1205005.68,100535.93,201.14
2,cama_mesa_banho,9417,1036988.68,204693.04,93.30
3,esporte_lazer,7720,988048.97,168607.51,114.34
4,informatica_acessorios,6689,911954.32,147318.08,116.51
5,moveis_decoracao,6449,729762.49,172749.30,87.56
6,cool_stuff,3632,635290.85,84039.10,167.36
7,utilidades_domesticas,5884,632248.66,146149.11,90.79
8,automotivo,3897,592720.11,92664.21,139.96
9,ferramentas_jardim,3518,485256.46,98962.75,111.63


**Interpretation:** поріг проходять 53 категорії з 73. Лідери beleza_saude і relogios_presentes. Найдорожчі товари в pcs (1 098 в середньому). Товари без категорії дали 179 535, це не так мало.

### 5.2. AOV по штатах (100+ замовлень)

In [ ]:
run("""
WITH order_totals AS (
    SELECT
        o.order_id,
        c.customer_state,
        SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.order_id,
        c.customer_state
)
SELECT
    customer_state,
    COUNT(*) AS n_orders,
    SUM(order_total)::NUMERIC(12, 2) AS gmv,
    (SUM(order_total) / NULLIF(COUNT(*), 0))::NUMERIC(8, 2) AS aov
FROM order_totals
GROUP BY customer_state
HAVING COUNT(*) >= 100
ORDER BY aov DESC NULLS LAST;
""")

,customer_state,n_orders,gmv,aov
0,PB,532,140987.81,265.01
1,AL,411,96229.40,234.13
2,RO,247,57558.02,233.03
3,PA,970,217647.11,224.38
4,TO,279,61354.42,219.91
5,PI,493,108132.28,219.34
6,SE,345,73032.32,211.69
7,RN,482,101895.08,211.40
8,CE,1327,275606.30,207.69
9,MT,903,186168.96,206.17


**Interpretation:** найвищий чек у PB (265), AL і RO, найнижчий у SP (143), хоча там найбільше замовлень. Скоріше за все через дорожчу доставку далеко від SP.

### 5.3. STRING_AGG по повторних покупцях

У прикладі `HAVING ... >= 1` нічого не фільтрує, тому поставив `>= 2`.

In [ ]:
run("""
SELECT
    c.customer_unique_id,
    COUNT(DISTINCT o.order_id) AS n_orders,
    SUM(oi.price)::NUMERIC(12, 2) AS total_spend,
    STRING_AGG(
        DISTINCT COALESCE(p.product_category_name, '(unknown)'),
        ', ' ORDER BY COALESCE(p.product_category_name, '(unknown)')
    ) AS categories
FROM olist_customers AS c
JOIN olist_orders AS o
    ON o.customer_id = c.customer_id
JOIN olist_order_items AS oi
    ON oi.order_id = o.order_id
LEFT JOIN olist_products AS p
    ON p.product_id = oi.product_id
GROUP BY c.customer_unique_id
HAVING COUNT(DISTINCT o.order_id) >= 2
ORDER BY total_spend DESC NULLS LAST
LIMIT 20;
""")

,customer_unique_id,n_orders,total_spend,categories
0,da122df9eeddfedc1dc1f5349a1a690c,2,7388.00,eletroportateis
1,c8460e4251689ba205045f3ea17884a1,4,4080.00,telefonia
2,055ec572ac7f3c7bdd04a183830ebe59,2,3999.98,automotivo
3,59d66d72939bc9497e19d89c61a96d5f,2,3459.00,esporte_lazer
4,58c1b085b54c03a1f1ab5f13d64c2b1c,2,2999.98,pcs
5,d77aa95864ae5b42160937615628723a,2,2400.00,pcs
6,7b0eaf68a16e4808e5388c67345033c9,2,2238.42,bebes
7,86df00dc5fd68f4dd5d5945ca19f3ed6,3,2200.00,informatica_acessorios
8,6ddbc64bd04d40f7768ff088d94cbeb8,2,2108.00,eletrodomesticos
9,73601b1eec55943e90ce8d61253d5c09,2,2009.98,consoles_games


In [ ]:
run("""
SELECT COUNT(*) AS repeat_customers
FROM (
    SELECT c.customer_unique_id
    FROM olist_customers AS c
    JOIN olist_orders AS o
        ON o.customer_id = c.customer_id
    GROUP BY c.customer_unique_id
    HAVING COUNT(DISTINCT o.order_id) >= 2
) AS t;
""")

,repeat_customers
0,2997


**Interpretation:** повторних покупців лише 2 997 (~3%). З тих, хто має товари (2 913), тільки 44% купують в одній категорії, тобто решта повертається за іншим товаром.

**WHERE vs HAVING:** WHERE фільтрує рядки до групування (наприклад, `WHERE oi.price > 100` змінив би суму по категорії). HAVING фільтрує вже готові групи, тому `SUM(oi.price) > 10000` можна писати тільки в HAVING.

## Завдання 6. Set operations і window

### 6.1. UNION

In [ ]:
run("""
SELECT customer_state AS state, 'customer' AS role
FROM olist_customers

UNION

SELECT seller_state AS state, 'seller' AS role
FROM olist_sellers
ORDER BY state, role;
""")

,state,role
0,AC,customer
1,AC,seller
2,AL,customer
3,AM,customer
4,AM,seller
5,AP,customer
6,BA,customer
7,BA,seller
8,CE,customer
9,CE,seller


**Interpretation:** UNION, а не UNION ALL, бо потрібні унікальні пари штат + роль. Вийшло 50 рядків: 27 для клієнтів і 23 для продавців.

### 6.2. INTERSECT

In [ ]:
run("""
SELECT customer_state AS state
FROM olist_customers

INTERSECT

SELECT seller_state AS state
FROM olist_sellers
ORDER BY state;
""")

,state
0,AC
1,AM
2,BA
3,CE
4,DF
5,ES
6,GO
7,MA
8,MG
9,MS


### 6.3. EXCEPT

In [ ]:
run("""
SELECT customer_state AS state
FROM olist_customers

EXCEPT

SELECT seller_state AS state
FROM olist_sellers
ORDER BY state;
""")

,state
0,AL
1,AP
2,RR
3,TO


**Interpretation:** INTERSECT дає 23 спільні штати, EXCEPT ті самі 4 штати, що й FULL OUTER JOIN.

### 6.4. ROW_NUMBER()

In [ ]:
run("""
WITH ranked_orders AS (
    SELECT
        c.customer_unique_id,
        o.customer_id,
        o.order_id,
        o.order_purchase_timestamp,
        ROW_NUMBER() OVER (
            PARTITION BY c.customer_unique_id
            ORDER BY o.order_purchase_timestamp DESC, o.order_id DESC
        ) AS rn
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
)
SELECT
    customer_unique_id,
    customer_id,
    order_id,
    order_purchase_timestamp
FROM ranked_orders
WHERE rn = 1
ORDER BY customer_unique_id
LIMIT 20;
""")

,customer_unique_id,customer_id,order_id,order_purchase_timestamp
0,0000366f3b9a7992bf8c76cfdf3221e2,fadbb3709178fc513abc1b2670aa1ad2,e22acc9c116caa3f2b7121bbb380d08e,2018-05-10 10:56:27
1,0000b849f77a49e4a4ce2b2a4ca5be3f,4cb282e167ae9234755102258dd52ee8,3594e05a005ac4d06a72673270ef9ec9,2018-05-07 11:11:27
2,0000f46a3911fa3c0805444483337064,9b3932a6253894a02c1df9d19004239f,b33ec3b699337181488304f362a6b734,2017-03-10 21:05:03
3,0000f6ccb0745a6a4b88665a16c9f078,914991f0c02ef0843c0e7010c819d642,41272756ecddd9a9ed0180413cc22fb6,2017-10-12 20:29:41
4,0004aac84e0df4da2b147fca70cf8255,47227568b10f5f58a524a75507e6992c,d957021f1127559cd947b62533f484f7,2017-11-14 19:45:42
5,0004bd2a26a76fe21f786e4fbd80607f,4a913a170c26e3c8052ed0202849b5a8,3e470077b690ea3e3d501cffb5e0c499,2018-04-05 19:33:16
6,00050ab1314c0e55a6ca13cf7181fecf,d2509c13692836fc0449e88cf9eb4858,d0028facea13f508e880202d7097a5a1,2018-04-20 12:57:23
7,00053a61a98854899e70ed204dd4bafe,a81ebb9b32f102298c0c89635b4b3154,44e608f2db00c74a1fe329de44416a4e,2018-02-28 11:15:41
8,0005e1862207bf6ccc02e4228effd9a0,3b37fb626fdf46cd99d37ec62afa88ff,ae76bef74b97bcb0b3e355e60d9a6f9c,2017-03-04 23:32:12
9,0005ef4cd20d2893f0d9fbd94d3c0d97,c59e8ff99836e90d8b457d4122dc34e9,01b330808c5819a6a3cb79b72f0b8288,2018-03-12 15:22:12


Перевірка з olist_customer_dim:

In [ ]:
run("""
WITH ranked_orders AS (
    SELECT
        c.customer_unique_id,
        o.customer_id,
        ROW_NUMBER() OVER (
            PARTITION BY c.customer_unique_id
            ORDER BY o.order_purchase_timestamp DESC, o.order_id DESC
        ) AS rn
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
)
SELECT
    COUNT(*) AS n_latest_orders,
    COUNT(DISTINCT r.customer_unique_id) AS n_customers,
    COUNT(*) FILTER (WHERE r.customer_id = cd.latest_customer_id) AS same_as_dim
FROM ranked_orders AS r
JOIN olist_customer_dim AS cd
    ON cd.customer_unique_id = r.customer_unique_id
WHERE r.rn = 1;
""")

,n_latest_orders,n_customers,same_as_dim
0,96096,96096,95961


**Interpretation:** вийшло рівно по одному рядку на покупця (96 096). 135 не збіглися з dim, бо в 290 випадках у покупця кілька замовлень з однаковим часом, а тай-брейкер різний (customer_id і order_id).

## Reflection

Найскладнішим для мене був не конкретний тип JOIN, а грануляність. Join через order_items пишеться легко, але множить відгуки: виходить 112 тис. замість 99 тис. Тому для seller_score спочатку звів дані до seller_id × order_id, а оцінки до одного значення на order_id. Інакше продавці з великими замовленнями отримали б завищену вагу відгуків. У FULL OUTER JOIN треба було не забути COALESCE для штату, бо з одного боку він NULL.

Найважче було придумати business-question для self join, бо пари продавців самі по собі мало що кажуть, і їх мільйони.

Для production-ETL найкориснішим вважаю ON CONFLICT: seller_score і customer_segments можна перезапускати без дублікатів. RETURNING зручний для логування змін.

Window functions я б використав для останнього замовлення покупця, номера покупки (перша чи повторна), днів між замовленнями через LAG і рангу продавців у категорії.

Проблеми в даних: 768 замовлень без відгуку, 610 товарів без категорії, 814 повторів review_id, 775 замовлень без позицій, 58 тис. відгуків без тексту, статуси created/approved, назва штату замість міста у продавців. Zip-код у схемі INTEGER, тому губляться нулі на початку.

customer_unique_id потрібен, бо customer_id новий на кожне замовлення, і тоді кожен покупець виглядав би новим. З ним видно 2 997 повторних покупців, і frequency в RFM має сенс. Для ML це важливо, бо ознаки мають описувати людину, а не окрему доставку.